# Lab 6 - Linear + Slack MCP Agent with Vaults

**Section 8 · Custom Tools & MCP**  ·  **Estimated time:** 30–40 min  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom on your own machine (you need a Linear workspace you control, and optionally a Slack workspace connection). Read the note above each cell, then run the cell with Shift+Enter.

> **Networking note:** the Managed Agents harness reaches MCP servers over the public internet. Always point at a public, remote, or SaaS MCP endpoint like the Linear one used here. Local MCP tunnels are a research preview and are not used in this course.

![Architecture](assets/architecture.svg)

## Goal
Build an agent that connects to Linear through the **public Linear MCP server** and can optionally connect to Slack through a second MCP server. Both connections are authenticated **per end user** with vault credentials. The agent uses Linear to triage open issues and file a new bug, while Slack can provide context or support follow-up actions. You will learn the two-step MCP wiring (declare each server on the agent, supply auth on the session via vaults), how to register per-user OAuth credentials, and how MCP tool calls and auth errors surface in the event stream.

## Prerequisites
- A **Linear** account and a workspace where you can create issues.
- A **Linear connection configured in Claude Managed Agents Vaults**. Paste `LINEAR_VAULT_ID` in the setup cell. The notebook can usually read the MCP URL from the vault credential; only set `LINEAR_MCP_URL` if the vault has multiple MCP credentials.
- Optional: a **Slack connection configured in Claude Managed Agents Vaults**. Set `SLACK_VAULT_ID` before running the agent if you want to attach Slack too.
- The public Linear MCP URL is usually `https://mcp.linear.app/mcp` - a hosted SaaS connector, so you do not run a local server or a tunnel.
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

print("Configure your Linear Managed Agents vault for this notebook.")
existing_vault_id = os.environ.get("LINEAR_VAULT_ID", "").strip()
if existing_vault_id.startswith("sk-ant-") or (existing_vault_id and not existing_vault_id.startswith("vlt_")):
    print("Clearing invalid LINEAR_VAULT_ID. It must be a vault id starting with 'vlt_', not an API key.")
    os.environ.pop("LINEAR_VAULT_ID", None)

if not os.environ.get("LINEAR_VAULT_ID"):
    os.environ["LINEAR_VAULT_ID"] = input(
        "Linear vault ID from Claude Console (vlt_...): "
    ).strip()

# Optional advanced override only. In the normal course path, leave this unset;
# the next cell reads the MCP URL from the vault credential.
# os.environ["LINEAR_MCP_URL"] = "https://mcp.linear.app/mcp"
# Optional Slack MCP vault, if your lab uses Slack too:
# os.environ["SLACK_VAULT_ID"] = "vlt_..."
# os.environ["SLACK_MCP_URL"] = "https://your-slack-mcp.example.com/mcp"

print("ANTHROPIC_API_KEY configured for this notebook session.")
print("LINEAR_VAULT_ID configured for this notebook session:", os.environ["LINEAR_VAULT_ID"])


In [ ]:
import os
from urllib.parse import urlparse
from anthropic import Anthropic

# The previous cell configures ANTHROPIC_API_KEY and LINEAR_VAULT_ID for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
DEFAULT_LINEAR_MCP_URL = "https://mcp.linear.app/mcp"

client = Anthropic()


def validate_vault_id(
    vault_id: str,
    env_var: str = "LINEAR_VAULT_ID",
    provider: str = "Linear",
) -> None:
    """Catch common copy/paste mistakes before sessions.create."""
    if not vault_id or vault_id.startswith("vlt_REPLACE"):
        raise RuntimeError(f"Set {env_var} to your Claude Managed Agents vault id.")
    if vault_id.startswith("sk-ant-"):
        raise RuntimeError(
            f"{env_var} currently contains an Anthropic API key. Paste the "
            f"{provider} vault id from Claude Console instead; it should start with 'vlt_'."
        )
    if not vault_id.startswith("vlt_"):
        raise RuntimeError(f"{env_var} should start with 'vlt_'. Got: {vault_id!r}")


def validate_mcp_url(
    url: str,
    url_env: str = "LINEAR_MCP_URL",
    vault_env: str = "LINEAR_VAULT_ID",
) -> None:
    """Catch missing or placeholder MCP URLs before agent creation."""
    parsed = urlparse(url)
    if not url or "REPLACE-ME" in url or parsed.scheme != "https" or not parsed.netloc:
        raise RuntimeError(
            f"Set {url_env} to a valid https MCP endpoint, or use a "
            f"{vault_env} whose credential contains an MCP server URL."
        )


def mcp_url_from_vault(vault_id: str, provider: str) -> str:
    """Read an MCP OAuth URL from an existing vault credential."""
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    mcp_credentials = [
        credential for credential in credentials
        if getattr(getattr(credential, "auth", None), "type", None) == "mcp_oauth"
    ]
    provider_credentials = [
        credential for credential in mcp_credentials
        if provider.lower() in (
            f"{getattr(credential, 'display_name', '') or ''} "
            f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '')}"
        ).lower()
    ]

    if len(provider_credentials) == 1:
        return provider_credentials[0].auth.mcp_server_url
    if len(mcp_credentials) == 1:
        return mcp_credentials[0].auth.mcp_server_url

    names = [
        f"{getattr(credential, 'display_name', '') or credential.id}: "
        f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '<no mcp url>')}"
        for credential in mcp_credentials
    ]
    raise RuntimeError(
        f"Could not uniquely identify the {provider} MCP credential in "
        f"vault {vault_id}. Set LINEAR_MCP_URL explicitly. "
        f"Found MCP credentials: {names or 'none'}"
    )


def resolve_optional_mcp_connection(provider: str, vault_env: str, url_env: str) -> tuple[str | None, str | None]:
    vault_id = os.environ.get(vault_env, "").strip()
    mcp_url = os.environ.get(url_env, "").strip()
    if not vault_id:
        return None, None

    validate_vault_id(vault_id, vault_env, provider.title())
    mcp_url = mcp_url or mcp_url_from_vault(vault_id, provider)
    validate_mcp_url(mcp_url, url_env, vault_env)
    return vault_id, mcp_url


LINEAR_VAULT_ID = os.environ.get("LINEAR_VAULT_ID", "").strip()
validate_vault_id(LINEAR_VAULT_ID, provider="Linear")
LINEAR_MCP_URL = os.environ.get("LINEAR_MCP_URL", "").strip() or mcp_url_from_vault(
    LINEAR_VAULT_ID, "linear"
)
validate_mcp_url(LINEAR_MCP_URL)
SLACK_VAULT_ID, SLACK_MCP_URL = resolve_optional_mcp_connection(
    "slack", "SLACK_VAULT_ID", "SLACK_MCP_URL"
)

print("SDK ready, model:", MODEL)
print("LINEAR_VAULT_ID:", LINEAR_VAULT_ID)
print("Linear MCP URL:", LINEAR_MCP_URL)
if SLACK_VAULT_ID:
    print("SLACK_VAULT_ID:", SLACK_VAULT_ID)
    print("Slack MCP URL:", SLACK_MCP_URL)


### Step 1 - Resolve vaults and declare MCP servers on the agent
The existing vault supplies auth at session time. This notebook reads the Linear MCP URL from the vault credential, then declares that MCP server **on the agent**. If `SLACK_VAULT_ID` is set, it uses the same pattern to attach Slack as a second MCP server. The `mcp_toolset` entries expose those tools to the model. We auto-approve Linear tool calls (`always_allow`) so the lab runs end to end - `always_ask` is the safer default in production.

In [ ]:
mcp_servers = [{
    "type": "url",
    "name": "linear",
    "url": LINEAR_MCP_URL,
}]
tools = [
    {"type": "agent_toolset_20260401"},
    {"type": "mcp_toolset", "mcp_server_name": "linear",
     "default_config": {"permission_policy": {"type": "always_allow"}}},
]
vault_ids = [LINEAR_VAULT_ID]

if SLACK_VAULT_ID and SLACK_MCP_URL:
    mcp_servers.append({
        "type": "url",
        "name": "slack",
        "url": SLACK_MCP_URL,
    })
    tools.append({"type": "mcp_toolset", "mcp_server_name": "slack"})
    vault_ids.append(SLACK_VAULT_ID)

agent = client.beta.agents.create(
    name="Linear Triage Agent",
    model=MODEL,
    system=(
        "You are a Linear assistant. You triage issues and file new ones "
        "via the linear MCP. When asked to file a bug, create a clear, "
        "well-titled issue with a concise description and reproduction "
        "steps. When asked to triage, list open or unassigned issues and "
        "suggest a priority for each. Always confirm the issue identifier "
        "(e.g. ENG-123) of anything you create or change. If a Slack MCP "
        "server is connected, use it only when the user explicitly asks "
        "for Slack context or Slack posting."
    ),
    mcp_servers=mcp_servers,
    tools=tools,
    betas=BETAS,
)
print("agent.id =", agent.id)

### Step 2 - Use the existing Linear vault
One agent definition serves many users. Each user's Linear or Slack credential lives in its **own vault**. In the course path, create/connect those credentials in Claude Console, then paste the resulting `LINEAR_VAULT_ID` in the setup cell and optionally set `SLACK_VAULT_ID`. OAuth tokens never appear in the notebook.

In [ ]:
print("Using existing Linear vault:", LINEAR_VAULT_ID)
print("Credential MCP URL:", LINEAR_MCP_URL)


### Step 3 - Credential registration happens in Claude Console
For this course flow, do not paste Linear or Slack access tokens into the notebook. Configure the SaaS connections in Claude Managed Agents Vaults, then use `LINEAR_VAULT_ID` and optionally `SLACK_VAULT_ID`. The script keeps a Linear OAuth-token fallback for automation, but the notebook demonstrates the safer vault-first path.

In [ ]:
# No raw OAuth token is needed in the notebook path.
# The session below attaches the prepared vault_ids list.


### Step 4 - Create an environment and a session that references the vault
The environment uses **limited** networking but allows MCP servers (`allow_mcp_servers: True`), so the harness can reach the public Linear endpoint. The session references the existing vault via `vault_ids`; auth wires up server-side.

In [ ]:
env = client.beta.environments.create(
    name="linear-env",
    config={"type": "cloud", "networking": {"type": "limited",
            "allow_mcp_servers": True, "allowed_hosts": []}},
    betas=BETAS,
)
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    vault_ids=vault_ids,
    title="Linear triage + file a bug",
    betas=BETAS,
)
print("env.id =", env.id)
print("session.id =", session.id)

### Step 5 - Ask the agent to triage and file an issue, then stream
Send one turn and consume the stream. Each Linear tool call surfaces as an `agent.mcp_tool_use` event; MCP auth problems surface as non-fatal `session.error` events.

In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{"type": "text",
            "text": ("Triage my unassigned Linear issues and tell me the top "
                     "three by likely priority. Then file a new bug titled "
                     "'Login crashes on empty password' with short reproduction "
                     "steps. Report the new issue identifier when done.")}],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "agent.mcp_tool_use":
            print(f"\n[mcp: {event.name}]")
        elif event.type == "session.error":
            print(f"\n[ERROR] {event.error}")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- `[mcp: linear_list_issues]` (or similar) as the agent reads your workspace.
- `[mcp: linear_create_issue]` as it files the new bug.
- The agent's final turn names the new issue identifier (for example `ENG-142`).
- A new issue titled "Login crashes on empty password" visible in your Linear workspace.
- No `session.error` events about MCP auth.

## Troubleshooting
- **`invalid vault_id`** → `LINEAR_VAULT_ID` must be the vault id from Claude Console and should start with `vlt_`. Do not paste your `sk-ant-...` API key into this field.
- **`session.error: mcp_auth_failed` or "missing credential"** → the vault credential is wrong, expired, or was not registered for the exact `mcp_server_url`. Confirm `LINEAR_VAULT_ID` points to the vault with the Linear MCP OAuth credential and that the credential URL matches the agent's `LINEAR_MCP_URL`.
- **Could not uniquely identify the Linear MCP credential** → the vault has multiple MCP credentials. Set `LINEAR_MCP_URL=https://mcp.linear.app/mcp` explicitly.
- **No MCP tool calls appear** → the agent did not discover the server. Confirm all three pieces are wired: the `mcp_servers` array on the agent, the `mcp_toolset` entry in `tools`, and `vault_ids` passed at session creation.
- **OAuth scope errors when creating issues** → the Linear connection in the vault lacks write scope. Reconnect or rotate the Linear credential in Claude Console with issue write access.
- **Tool calls pause for confirmation** → the MCP default permission policy is `always_ask`. Either set `always_allow` on the `mcp_toolset` (Step 1) or implement the confirmation flow to approve each call. Leaving it as `always_ask` is the safer default in production.
- **Networking blocked** → make sure the environment has `allow_mcp_servers: True`; with strict networking the harness cannot reach the public Linear endpoint.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste these prompts in order. Have your Linear vault id ready.

1. > Create a Managed Agents agent named "Linear Triage Agent" on model `claude-haiku-4-5-20251001`. Declare a single MCP server named `linear` with URL `https://mcp.linear.app/mcp`. Give it the toolset `agent_toolset_20260401` plus an `mcp_toolset` for `linear` with permission policy `always_allow`. The system prompt should tell it to triage and file Linear issues and always confirm the issue identifier it creates.

2. > Use my existing Linear Managed Agents vault from `LINEAR_VAULT_ID`. Read the Linear MCP URL from the vault credential when `LINEAR_MCP_URL` is unset, then declare that URL on the agent.

3. > Create a cloud environment named `linear-env` with limited networking but `allow_mcp_servers` true. Then start a session for the agent on that environment, passing the Linear vault id, plus Slack if configured, in `vault_ids`. Title it "Linear triage + file a bug".

4. > In that session, send this user message and stream the response, printing each `agent.mcp_tool_use` name as it happens: "Triage my unassigned Linear issues and tell me the top three by likely priority. Then file a new bug titled 'Login crashes on empty password' with short reproduction steps. Report the new issue identifier when done." Stop when the session goes idle.

## Stretch
- Have the agent **label and assign** the issue it files (for example label `bug`, assign to yourself), then confirm the change with another MCP tool call.
- Ask it to **query existing issues** by team or status and produce a short triage report before filing anything.
- Reconnect or rotate the Linear credential in Claude Console and confirm a new session picks up the refreshed secret.
- Switch the `mcp_toolset` policy to `always_ask` and implement the approval flow so a human confirms each write to Linear.